# 03 DeBERTa-v3-small 训练

对齐 `code/deberta_train_exp5.py`：
- 读取 `plies-and-ultra/*.parquet`
- 可选融合 `human_llm_parquet`
- 训练 DeBERTa-v3-small 二分类


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import roc_auc_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

In [ ]:
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

MODEL_CHECKPOINT = 'microsoft/deberta-v3-small'
PILE_DIR = ROOT / 'plies-and-ultra'
HUMAN_LLM_PARQUET = None
VALID_CSV = ROOT / 'code/nonTargetText_llm_slightly_modified_gen.csv'
OUTPUT_DIR = ROOT / 'models/deberta/deberta-v3-small-finetuned_v5_notebook'

MAX_LENGTH = 384
BATCH_SIZE = 16
NUM_EPOCHS = 6.0
LR = 1e-4
WEIGHT_DECAY = 0.01
PATIENCE = 3
SEED = 42

print('PILE_DIR =', PILE_DIR)

In [ ]:
def softmax_np(logits):
    x = logits - np.max(logits, axis=-1, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=-1, keepdims=True)


def load_train_from_parquets(pile_dir: Path, human_llm_parquet: Path | None, seed: int) -> pd.DataFrame:
    dfs = []

    for name in ['pile2.parquet', 'plies3.parquet', 'plies4.parquet', 'Ultra.parquet', 'lmsys.parquet']:
        p = pile_dir / name
        if p.exists():
            dfs.append(pd.read_parquet(p, engine='fastparquet'))

    if human_llm_parquet is not None and Path(human_llm_parquet).exists():
        human_llm = pd.read_parquet(human_llm_parquet, engine='fastparquet')
        if {'text', 'source'}.issubset(human_llm.columns):
            human_llm['label'] = np.where(human_llm['source'] == 'Human', 0, 1)
            dfs.append(human_llm[['text', 'label']])

    if not dfs:
        raise FileNotFoundError('No training parquet loaded.')

    train = pd.concat(dfs, axis=0, ignore_index=True)
    if not {'text', 'label'}.issubset(train.columns):
        raise ValueError('Training data must contain text,label columns')

    train['text'] = train['text'].fillna('').astype(str).str.strip('\n')
    train = train.dropna(subset=['text', 'label']).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    train['label'] = train['label'].astype(int)
    return train


train = load_train_from_parquets(PILE_DIR, HUMAN_LLM_PARQUET, SEED)
valid = pd.read_csv(VALID_CSV)[['text', 'label']].copy()
valid['text'] = valid['text'].fillna('').astype(str).str.strip('\n')
valid['label'] = valid['label'].astype(int)

print('train:', train.shape, 'valid:', valid.shape)

## 数据分布

训练前先检查样本标签与长度。

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='talk')
PLOT_DIR = OUTPUT_DIR / 'plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
train_counts = train['label'].value_counts().sort_index()
valid_counts = valid['label'].value_counts().sort_index()
ax[0].bar(train_counts.index.astype(str), train_counts.values, color=['#4c72b0', '#55a868'])
ax[0].set_title('DeBERTa Train Label Distribution')
ax[1].bar(valid_counts.index.astype(str), valid_counts.values, color=['#4c72b0', '#55a868'])
ax[1].set_title('DeBERTa Valid Label Distribution')
fig.tight_layout()
fig.savefig(PLOT_DIR / 'deberta_label_distribution.png', dpi=220)
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(train['text'].str.len(), bins=60, stat='density', alpha=0.35, label='train', ax=ax)
sns.histplot(valid['text'].str.len(), bins=60, stat='density', alpha=0.35, label='valid', ax=ax)
ax.set_title('DeBERTa Text Length Distribution')
ax.set_xlabel('Character Length')
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / 'deberta_text_length_distribution.png', dpi=220)
plt.show()

In [ ]:
ds_train = Dataset.from_pandas(train[['text', 'label']])
ds_valid = Dataset.from_pandas(valid[['text', 'label']])

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def preprocess(examples):
    return tokenizer(examples['text'], max_length=MAX_LENGTH, padding=True, truncation=True)

ds_train_enc = ds_train.map(preprocess, batched=True)
ds_valid_enc = ds_valid.map(preprocess, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=2)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model.to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax_np(logits)
    auc = roc_auc_score(labels, probs[:, 1], multi_class='ovr')
    return {'roc_auc': auc}

In [ ]:
train_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=LR,
    lr_scheduler_type='cosine',
    fp16=torch.cuda.is_available(),
    optim='adamw_torch',
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model='roc_auc',
    report_to='none',
    save_total_limit=3,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=ds_train_enc,
    eval_dataset=ds_valid_enc,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print('saved:', OUTPUT_DIR)

## 训练后评估

训练完成后立即绘制 loss/AUC/ROC/混淆矩阵。

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix

sns.set_theme(style='whitegrid', context='talk')
PLOT_DIR = OUTPUT_DIR / 'plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

log_df = pd.DataFrame(trainer.state.log_history)
display(log_df.tail(10))

# 1) 训练/验证 loss
fig, ax = plt.subplots(figsize=(10, 6))
if 'loss' in log_df.columns:
    x_train = log_df.loc[log_df['loss'].notna(), 'step']
    y_train = log_df.loc[log_df['loss'].notna(), 'loss']
    ax.plot(x_train, y_train, label='train loss', lw=2)
if 'eval_loss' in log_df.columns:
    x_eval = log_df.loc[log_df['eval_loss'].notna(), 'step']
    y_eval = log_df.loc[log_df['eval_loss'].notna(), 'eval_loss']
    ax.plot(x_eval, y_eval, label='eval loss', lw=2)
ax.set_title('DeBERTa-v3-small Training Curve (Loss)')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / 'deberta_loss_curve.png', dpi=220)
plt.show()

# 2) 验证 AUC 曲线
if 'eval_roc_auc' in log_df.columns and log_df['eval_roc_auc'].notna().any():
    fig, ax = plt.subplots(figsize=(10, 6))
    x = log_df.loc[log_df['eval_roc_auc'].notna(), 'epoch']
    y = log_df.loc[log_df['eval_roc_auc'].notna(), 'eval_roc_auc']
    ax.plot(x, y, marker='o', lw=2, color='#55a868')
    ax.set_title('DeBERTa Validation AUC by Epoch')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('ROC-AUC')
    fig.tight_layout()
    fig.savefig(PLOT_DIR / 'deberta_auc_curve.png', dpi=220)
    plt.show()

# 3) 验证 ROC + 混淆矩阵
pred_out = trainer.predict(ds_valid_enc)
logits = pred_out.predictions
labels = pred_out.label_ids
probs = softmax_np(logits)[:, 1]

fpr, tpr, _ = roc_curve(labels, probs)
roc_auc = auc(fpr, tpr)
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, lw=2, label=f'AUC={roc_auc:.4f}')
ax.plot([0, 1], [0, 1], '--', color='gray')
ax.set_title('DeBERTa ROC on Validation Set')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
fig.tight_layout()
fig.savefig(PLOT_DIR / 'deberta_valid_roc.png', dpi=220)
plt.show()

cm = confusion_matrix(labels, (probs >= 0.5).astype(int))
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=ax)
ax.set_title('DeBERTa Confusion Matrix @0.5')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
fig.tight_layout()
fig.savefig(PLOT_DIR / 'deberta_valid_confusion_matrix.png', dpi=220)
plt.show()

print('plots saved to', PLOT_DIR)